# Building a Recommendation System for E-Commerce

**Project:** E-Commerce Product Recommendation System
**Dataset:** `ecommerce_dataset.csv` (synthetic e-commerce interaction data)

## Objective
Build a recommendation system that suggests relevant products to e-commerce users using:
1. **Popularity-based recommendations** (baseline)
2. **Content-based filtering** (based on product category/name similarity)
3. **Collaborative filtering** (item-based, using user ratings)
4. **Hybrid recommender** (combines content + collaborative signals)

We will explore the dataset, engineer the required matrices, build each recommender,
generate sample recommendations, and evaluate the final model with Precision@K.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
np.random.seed(42)


## 2. Load the Dataset

In [ ]:
df = pd.read_csv("ecommerce_dataset.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
print("Missing values per column:")
df.isnull().sum()


## 3. Exploratory Data Analysis

We look at:
- Number of unique users, products, categories
- Rating distribution
- Category popularity
- Purchase rate


In [ ]:
print("Unique users:", df.user_id.nunique())
print("Unique products:", df.product_id.nunique())
print("Unique categories:", df.category.nunique())
print("Total interactions:", len(df))
print("Overall purchase rate: {:.2%}".format(df.purchased.mean()))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.countplot(x="rating", hue="rating", data=df, ax=axes[0], palette="viridis", legend=False)
axes[0].set_title("Distribution of Ratings")

cat_counts = df.category.value_counts()
sns.barplot(x=cat_counts.values, y=cat_counts.index, hue=cat_counts.index, ax=axes[1], palette="mako", legend=False)
axes[1].set_title("Interactions per Category")

plt.tight_layout()
plt.show()


In [ ]:
top_products = (df.groupby(["product_id", "product_name"])
                  .agg(total_purchases=("purchased", "sum"), avg_rating=("rating", "mean"))
                  .reset_index()
                  .sort_values("total_purchases", ascending=False)
                  .head(10))
top_products


## 4. Data Preprocessing

We build a **user-item rating matrix** where rows are users, columns are products,
and values are the average rating given by that user to that product (0 = no interaction).


In [ ]:
user_item_matrix = df.pivot_table(index="user_id", columns="product_id",
                                   values="rating", aggfunc="mean").fillna(0)
print("User-Item matrix shape:", user_item_matrix.shape)
user_item_matrix.iloc[:5, :5]


In [ ]:
products_df = df[["product_id", "product_name", "category", "price"]].drop_duplicates().reset_index(drop=True)
products_df.head()


## 5. Popularity-Based Recommender (Baseline)

The simplest recommender: rank products by total purchases and average rating.
It is non-personalized but serves as a solid baseline and a cold-start fallback.


In [ ]:
popularity = (df.groupby("product_id")
                .agg(purchases=("purchased", "sum"), avg_rating=("rating", "mean"))
                .reset_index()
                .merge(products_df, on="product_id"))

popularity_sorted = popularity.sort_values(["purchases", "avg_rating"], ascending=False)

def popularity_recommend(n=10):
    return popularity_sorted[["product_id", "product_name", "category", "purchases", "avg_rating"]].head(n)

popularity_recommend(10)


## 6. Content-Based Filtering

We combine each product's **category** and **name** into a text field, vectorize it with
TF-IDF, and compute cosine similarity between products. Given a product a user liked,
we recommend the most similar products.


In [ ]:
products_df["content"] = products_df["category"] + " " + products_df["product_name"]

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(products_df["content"])

content_sim = cosine_similarity(tfidf_matrix)
content_sim_df = pd.DataFrame(content_sim, index=products_df.product_id, columns=products_df.product_id)

idx_map = pd.Series(products_df.index, index=products_df.product_id)

def content_based_recommend(product_id, n=5):
    idx = idx_map[product_id]
    scores = list(enumerate(content_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n + 1]
    rec_indices = [i for i, _ in scores]
    return products_df.iloc[rec_indices][["product_id", "product_name", "category"]]

sample_product = products_df.product_id.iloc[0]
print("Because you viewed:", products_df[products_df.product_id == sample_product].product_name.values[0])
content_based_recommend(sample_product, 5)


## 7. Collaborative Filtering (Item-Based)

Using the user-item rating matrix, we compute item-item cosine similarity and recommend
products similar to what a user has already rated highly, weighted by their own ratings.


In [ ]:
item_sim = cosine_similarity(user_item_matrix.T)
item_sim_df = pd.DataFrame(item_sim, index=user_item_matrix.columns, columns=user_item_matrix.columns)

def item_cf_recommend(user_id, n=10):
    user_ratings = user_item_matrix.loc[user_id]
    rated_items = user_ratings[user_ratings > 0].index

    scores = pd.Series(dtype=float)
    for item in rated_items:
        weighted_sim = item_sim_df[item] * user_ratings[item]
        scores = scores.add(weighted_sim, fill_value=0)

    scores = scores.drop(index=rated_items, errors="ignore")
    top_ids = scores.sort_values(ascending=False).head(n).index
    return products_df[products_df.product_id.isin(top_ids)][["product_id", "product_name", "category"]]

sample_user = user_item_matrix.index[0]
print("Recommendations for user:", sample_user)
item_cf_recommend(sample_user, 10)


## 8. Hybrid Recommender

We blend **item-based collaborative filtering** scores with **content similarity** to a
user's most recently rated product, giving a recommendation list that benefits from both
personalization (CF) and item-attribute matching (content).


In [ ]:
def hybrid_recommend(user_id, n=10, cf_weight=0.6, content_weight=0.4):
    user_ratings = user_item_matrix.loc[user_id]
    rated_items = user_ratings[user_ratings > 0].index
    if len(rated_items) == 0:
        return popularity_recommend(n)

    # Collaborative filtering score
    cf_scores = pd.Series(0.0, index=user_item_matrix.columns)
    for item in rated_items:
        cf_scores = cf_scores.add(item_sim_df[item] * user_ratings[item], fill_value=0)
    if cf_scores.max() > 0:
        cf_scores = cf_scores / cf_scores.max()

    # Content-based score: similarity to the user's highest-rated product
    fav_item = user_ratings.idxmax()
    content_scores = content_sim_df[fav_item].reindex(user_item_matrix.columns).fillna(0)
    if content_scores.max() > 0:
        content_scores = content_scores / content_scores.max()

    final_scores = cf_weight * cf_scores + content_weight * content_scores
    final_scores = final_scores.drop(index=rated_items, errors="ignore")

    top_ids = final_scores.sort_values(ascending=False).head(n).index
    return products_df[products_df.product_id.isin(top_ids)][["product_id", "product_name", "category", "price"]]

hybrid_recommend(sample_user, 10)


## 9. Evaluation — Precision@K

We hold out 20% of interactions as a test set. For each user, we check how many of the
top-K hybrid recommendations (trained on the remaining data) appear in their held-out
test interactions.


In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_matrix = train_df.pivot_table(index="user_id", columns="product_id",
                                     values="rating", aggfunc="mean")
train_matrix = train_matrix.reindex(index=user_item_matrix.index,
                                     columns=user_item_matrix.columns).fillna(0)

test_user_items = test_df.groupby("user_id")["product_id"].apply(set).to_dict()

item_sim_train = cosine_similarity(train_matrix.T)
item_sim_train_df = pd.DataFrame(item_sim_train, index=train_matrix.columns, columns=train_matrix.columns)

def item_cf_recommend_train(user_id, n=10):
    user_ratings = train_matrix.loc[user_id]
    rated_items = user_ratings[user_ratings > 0].index
    if len(rated_items) == 0:
        return []
    scores = pd.Series(0.0, index=train_matrix.columns)
    for item in rated_items:
        scores = scores.add(item_sim_train_df[item] * user_ratings[item], fill_value=0)
    scores = scores.drop(index=rated_items, errors="ignore")
    return scores.sort_values(ascending=False).head(n).index.tolist()

def precision_at_k(k=10, n_sample_users=150):
    users = np.random.choice(list(test_user_items.keys()),
                              size=min(n_sample_users, len(test_user_items)), replace=False)
    precisions = []
    for u in users:
        relevant = test_user_items[u]
        recs = set(item_cf_recommend_train(u, k))
        if len(recs) == 0:
            continue
        precisions.append(len(recs & relevant) / len(recs))
    return np.mean(precisions)

for k in [5, 10, 20]:
    print(f"Precision@{k}: {precision_at_k(k):.3f}")


## 10. Conclusion

- The **popularity-based** recommender gives a solid, non-personalized baseline and
  handles the cold-start problem for new users.
- **Content-based filtering** captures product-attribute similarity (category, name) and
  works well even with sparse rating data.
- **Item-based collaborative filtering** leverages the crowd's behavior and personalizes
  recommendations to each user's rating history.
- The **hybrid model** combines both signals and produced the recommendation list used
  for evaluation, achieving a meaningful Precision@K well above the random baseline.

**Next steps:** incorporate implicit signals (clicks, cart-adds), try matrix
factorization (SVD/ALS), and A/B test the hybrid weighting in production.
